# RSNA Knee — submission

A launcher, not a pipeline. It mounts three things and calls one function:

| mount | what it is |
|---|---|
| `rsna-src` | our library, published as a dataset — internet is off, so this is the only channel |
| `rsna-knee-weights` | the trained package: `manifest.json` plus one `.pt` per fold |
| `metaresearch/dinov2` | the encoder, hosted by Kaggle. We do not publish our own copy |

Every decision — resolution, slice count, slots, stem, encoder variant — comes from the
package's own manifest. Nothing about the model is written here, so this notebook runs
**any** model we ever train without being edited.

The chain itself is `rsna.infer.run_submission`, which `python -m scripts.predict`
calls too. One implementation, so a leaderboard score is reproducible locally.


In [ ]:
import os, sys, tempfile, time, zipfile
from pathlib import Path

T0 = time.time()
INPUT = Path("/kaggle/input")


def log(message):
    print(f"[{time.time() - T0:7.1f}s] {message}", flush=True)


def walk(root):
    """Every directory under `root`, skipping the two huge DICOM trees."""
    for current, dirs, files in os.walk(root):
        dirs[:] = [d for d in dirs if d not in ("train_series", "test_series")]
        yield Path(current), dirs, files


def find(marker, contains=None):
    """First directory holding `marker`.

    Searched by content rather than by path: Kaggle names a mount after its dataset,
    and a rename would otherwise break this notebook in silence.
    """
    for current, _, files in walk(INPUT):
        if marker in files and (contains is None or contains in str(current)):
            return current
    return None


def find_library():
    """The package root, under whatever name the mount happens to use.

    Three shapes have to be handled, and which one arrives is Kaggle's decision, not
    ours: an expanded `rsna/` tree, the same tree expanded *without* its top level
    (which is what a zipped upload actually produces — the directory name is dropped),
    or an unexpanded `rsna.zip`.
    """
    for current, dirs, files in walk(INPUT):
        if "config.py" in files and "infer" in dirs:
            return current
    for current, _, files in walk(INPUT):
        for name in files:
            if name.endswith(".zip") and "rsna" in name:
                target = Path(tempfile.mkdtemp())
                with zipfile.ZipFile(current / name) as archive:
                    archive.extractall(target)
                log(f"library extracted from {current / name}")
                for inner, dirs, files in os.walk(target):
                    if "config.py" in files and "infer" in dirs:
                        return Path(inner)
    return None


library = find_library()
if library is None:
    raise SystemExit("the rsna source dataset is not mounted")

# The import name is `rsna` whatever the mount is called, so it is given that name
# here rather than assumed. A symlink, so nothing is copied.
staging = Path(tempfile.mkdtemp())
os.symlink(library, staging / "rsna")
sys.path.insert(0, str(staging))
log(f"library: {library}")

root = find("test.csv")
if root is None:
    raise SystemExit("the competition data is not mounted")
log(f"competition data: {root}")


In [ ]:
import json

import torch

from rsna.config import Config
from rsna.infer import run_submission
from rsna.model import find_encoder
from rsna.model.package import find_package

package = find_package(INPUT)
if package is None:
    raise SystemExit("no weights package is mounted")

manifest = json.loads((package / "manifest.json").read_text())
config = Config.from_dict(manifest["members"][0]["config"])
encoder = find_encoder(config, INPUT)
if encoder is None:
    raise SystemExit(f"no {config.encoder}-{config.encoder_variant} encoder is mounted")

device = "cuda" if torch.cuda.is_available() else "cpu"
log(f"package: {package}  ({len(manifest['members'])} member(s))")
log(f"encoder: {encoder}")
log(f"device: {device}" + (f"  {torch.cuda.get_device_name(0)}" if device == "cuda" else ""))

# No try/except on purpose. A run that swallows its failure writes the 0.5 fallback and
# reports COMPLETE, which is indistinguishable from a model that learnt nothing — we
# have already been caught by exactly that. Let it fail loudly instead.
run_submission(package, root, "test_series", encoder=encoder,
               out="submission.csv", device=device, log=log)


In [ ]:
import pandas as pd

from rsna.config import TARGETS

submission = pd.read_csv("submission.csv", dtype={"StudyInstanceUID": str})
assert list(submission.columns) == ["StudyInstanceUID"] + TARGETS, "column names"
assert submission["StudyInstanceUID"].is_unique, "one row per study"
assert submission[TARGETS].notna().all().all(), "no missing predictions"

distinct = submission[TARGETS].to_numpy().round(9)
log(f"submission.csv: {submission.shape}, {len(set(distinct.ravel()))} distinct values")
submission.head()
